In [3]:
"""
figures_daf_audit.py

Generates:
  1) Radar plot for each dataset (6 metrics)
  2) Cross-dataset 8x6 heatmap of metric scores
  3) Metric ranking plot (barplots per metric)
  4) Gap analysis figure (deficient/adequate/strong heatmap)

Dependencies:
  pip install numpy pandas matplotlib seaborn
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ---------- 1) Data (from user-supplied scores) ----------
datasets = ["NSL-KDD","ToN-IoT","UNSW-NB15","HIKARI-2021",
            "CIC-IDS2017","BoT-IoT","CSE-CIC-IDS2018","UNR-IDD"]

data = {
    "Dataset": datasets,
    "ATC":      [0.4226,0.3916,0.3449,0.3102,0.2845,0.2625,0.2397,0.2301],
    "AS":       [0.2264,0.3671,0.2037,0.1923,0.1387,0.1645,0.2290,0.1994],
    "FCPA":     [0.5952,0.5536,0.3524,0.7143,0.6429,0.5071,0.6429,0.6333],
    "PDR":      [0.728092,0.729147,0.598785,0.636033,0.656423,0.817848,0.701781,np.nan],
    "CBA":      [0.634779,0.854908,0.779950,0.630695,0.543060,0.632958,0.559353,0.832675],
    "TRI":      [0.517023,0.457623,0.477142,0.479349,0.531016,0.430933,0.528349,0.760132]
}

df = pd.DataFrame(data).set_index("Dataset")
metrics = ["ATC","AS","FCPA","PDR","CBA","TRI"]

# create output dir
outdir = "figures"
os.makedirs(outdir, exist_ok=True)

# ---------- Utility: style ----------
sns.set(style="whitegrid")
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10
})

# ---------- 1) Radar plots (one per dataset) ----------
def radar_plot(values, labels, title, filename):
    """
    values: list of metric values (len = N)
    labels: list of metric names
    """
    N = len(labels)
    # angles for each axis
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    vals = values + values[:1]

    fig, ax = plt.subplots(figsize=(4.2,4.2), subplot_kw=dict(polar=True))
    ax.plot(angles, vals, color="#DD8500", linewidth=2)
    ax.fill(angles, vals, color="#FFD8A8", alpha=0.35)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_yticks([0.1,0.3,0.5,0.7,0.9])
    ax.set_ylim(0,1)
    ax.set_title(title, y=1.08)
    # grid, style
    ax.grid(color="gray", linestyle="--", linewidth=0.6, alpha=0.5)
    plt.tight_layout()
    fig.savefig(filename, bbox_inches="tight")
    plt.close(fig)

# generate radar plots
for ds in df.index:
    vals = df.loc[ds, metrics].fillna(0).tolist()
    fname = os.path.join(outdir, f"radar_{ds.replace('/','_')}.png")
    radar_plot(vals, metrics, ds, fname)

print(f"Saved {len(df.index)} radar plots to '{outdir}/'")

# ---------- 2) Cross-dataset heatmap (8x6) ----------
plt.figure(figsize=(8,5))
hm_df = df[metrics].copy()
# reorder rows to original order (already)
# annotate with values rounded
sns.heatmap(hm_df, annot=True, fmt=".3f", cmap="YlOrBr", cbar_kws={'label':'score'}, linewidths=0.5)
plt.title("Cross-dataset Metric Scores (8 × 6)")
plt.xlabel("Metrics")
plt.ylabel("Dataset")
plt.tight_layout()
plt.savefig(os.path.join(outdir,"heatmap_8x6_metrics.png"), bbox_inches="tight")
plt.close()
print("Saved cross-dataset heatmap.")

# ---------- 3) Metric ranking plot ----------
# For each metric, plot datasets sorted by score (stacked small multiples)
n_metrics = len(metrics)
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12,7))
axes = axes.flatten()
for i, m in enumerate(metrics):
    ax = axes[i]
    series = df[m].sort_values(ascending=True)  # low->high
    series.plot(kind="barh", ax=ax, color="#3388CC")
    ax.set_xlim(0,1)
    ax.set_title(m)
    ax.set_xlabel("Score")
    for p in ax.patches:
        w = p.get_width()
        ax.text(w + 0.01, p.get_y() + p.get_height()/2, f"{w:.3f}", va='center', fontsize=8)
plt.tight_layout()
fig.suptitle("Metric rankings across datasets (per-metric sort)", y=1.02, fontsize=14)
plt.savefig(os.path.join(outdir,"metric_rankings.png"), bbox_inches="tight")
plt.close()
print("Saved metric ranking plot.")

# ---------- 4) Gap analysis figure ----------
# thresholds: deficient <0.4, adequate 0.4-0.7, strong >0.7
def gap_category(x):
    if pd.isna(x):
        return -1  # special code for "N/A"
    if x < 0.4:
        return 0  # deficient
    elif x < 0.7:
        return 1  # adequate
    else:
        return 2  # strong

gap_df = df[metrics].map(gap_category)

# map to colors
cmap = { -1: (0.9,0.9,0.9), 0: (0.85,0.15,0.15), 1: (0.95,0.8,0.2), 2: (0.2,0.6,0.2) }
# create color matrix
color_matrix = gap_df.replace(cmap)

# plot matrix of colored rectangles with labels
fig, ax = plt.subplots(figsize=(8,4.5))
for i, ds in enumerate(gap_df.index):
    for j, m in enumerate(gap_df.columns):
        cat = gap_df.iloc[i, j]
        color = cmap.get(cat, (0.9,0.9,0.9))
        rect = plt.Rectangle((j, len(gap_df)-1 - i), 1, 1, facecolor=color, edgecolor="white")
        ax.add_patch(rect)
        # add the numeric score inside (or 'NA')
        val = df.iloc[i][m]
        text = "NA" if pd.isna(val) else f"{val:.2f}"
        ax.text(j + 0.5, len(gap_df)-1 - i + 0.5, text, ha='center', va='center', color='black', fontsize=9)
# set ticks and labels
ax.set_xticks(np.arange(len(gap_df.columns)) + 0.5)
ax.set_xticklabels(gap_df.columns, rotation=45)
ax.set_yticks(np.arange(len(gap_df.index)) + 0.5)
ax.set_yticklabels(list(reversed(list(gap_df.index))))
ax.set_xlim(0, len(gap_df.columns))
ax.set_ylim(0, len(gap_df.index))
ax.invert_yaxis()
ax.set_title("Gap Analysis (Deficient / Adequate / Strong)\nColors: Red=Deficient, Yellow=Adequate, Green=Strong, Gray=N/A")
plt.tight_layout()
plt.savefig(os.path.join(outdir,"gap_analysis.png"), bbox_inches="tight")
plt.close()
print("Saved gap analysis figure.")

print("All figures generated and saved in the 'figures' folder.")


Saved 8 radar plots to 'figures/'
Saved cross-dataset heatmap.
Saved metric ranking plot.
Saved gap analysis figure.
All figures generated and saved in the 'figures' folder.
